Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading Dataset Into Colab from Drive

In [ ]:
import pandas as pd
import numpy as np

df = pd.read_csv(
    '/content/drive/MyDrive/swiggy.csv',
    encoding='latin1',
    engine='python',
    on_bad_lines='skip'
)

df.head()

,url,address,name,online_order,book_table,rate,votes,phone,location,rest_type,dish_liked,cuisines,approx_cost(for two people),reviews_list,menu_item,listed_in(type),listed_in(city)
0,https://www.zomato.com/bangalore/jalsa-banasha...,"942, 21st Main Road, 2nd Stage, Banashankari, ...",Jalsa,Yes,Yes,4.1/5,775,080 42297555\r\n+91 9743772233,Banashankari,Casual Dining,"Pasta, Lunch Buffet, Masala Papad, Paneer Laja...","North Indian, Mughlai, Chinese",800,"[('Rated 4.0', 'RATED\n A beautiful place to ...",[],Buffet,Banashankari
1,https://www.zomato.com/bangalore/spice-elephan...,"2nd Floor, 80 Feet Road, Near Big Bazaar, 6th ...",Spice Elephant,Yes,No,4.1/5,787,080 41714161,Banashankari,Casual Dining,"Momos, Lunch Buffet, Chocolate Nirvana, Thai G...","Chinese, North Indian, Thai",800,"[('Rated 4.0', 'RATED\n Had been here for din...",[],Buffet,Banashankari
2,https://www.zomato.com/SanchurroBangalore?cont...,"1112, Next to KIMS Medical College, 17th Cross...",San Churro Cafe,Yes,No,3.8/5,918,+91 9663487993,Banashankari,"Cafe, Casual Dining","Churros, Cannelloni, Minestrone Soup, Hot Choc...","Cafe, Mexican, Italian",800,"[('Rated 3.0', ""RATED\n Ambience is not that ...",[],Buffet,Banashankari
3,https://www.zomato.com/bangalore/addhuri-udupi...,"1st Floor, Annakuteera, 3rd Stage, Banashankar...",Addhuri Udupi Bhojana,No,No,3.7/5,88,+91 9620009302,Banashankari,Quick Bites,Masala Dosa,"South Indian, North Indian",300,"[('Rated 4.0', ""RATED\n Great food and proper...",[],Buffet,Banashankari
4,https://www.zomato.com/bangalore/grand-village...,"10, 3rd Floor, Lakshmi Associates, Gandhi Baza...",Grand Village,No,No,3.8/5,166,+91 8026612447\r\n+91 9901210005,Basavanagudi,Casual Dining,"Panipuri, Gol Gappe","North Indian, Rajasthani",600,"[('Rated 4.0', 'RATED\n Very good restaurant ...",[],Buffet,Banashankari


Checking Dataset for Preprocessing

In [ ]:
df = df[['name', 'location', 'cuisines', 'rate', 'votes', 'approx_cost(for two people)']]

df.head()

,name,location,cuisines,rate,votes,approx_cost(for two people)
0,Jalsa,Banashankari,"North Indian, Mughlai, Chinese",4.1/5,775,800
1,Spice Elephant,Banashankari,"Chinese, North Indian, Thai",4.1/5,787,800
2,San Churro Cafe,Banashankari,"Cafe, Mexican, Italian",3.8/5,918,800
3,Addhuri Udupi Bhojana,Banashankari,"South Indian, North Indian",3.7/5,88,300
4,Grand Village,Basavanagudi,"North Indian, Rajasthani",3.8/5,166,600


Checking if there are null values

In [ ]:
df.isnull().sum()

,0
name,0
location,21
cuisines,45
rate,7775
votes,0
approx_cost(for two people),344


Dropping Unnecessary Columns

In [ ]:
df.dropna(inplace=True)

Data Cleaning and Preprocessing – Rating Column

In [ ]:
df['rate'] = df['rate'].str.replace('/5', '')
df = df[df['rate'] != 'NEW']
df = df[df['rate'] != '-']

df['rate'] = df['rate'].astype(float)

Data Cleaning and Preprocessing – Cost Column

In [ ]:
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(str)
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].str.replace(',', '')
df['approx_cost(for two people)'] = df['approx_cost(for two people)'].astype(float)

Data Cleaning and Preprocessing – Cuisines Column

In [ ]:
df['cuisines'] = df['cuisines'].str.lower()

Dataset Overview and Structure

In [ ]:
df.head()
df.shape

(40815, 6)

Recommendation Function (Filtering and Ranking)

In [ ]:
def recommend_restaurants(location, cuisine, max_cost):

    # Filter based on user input
    filtered = df[
        (df['location'].str.lower() == location.lower()) &
        (df['cuisines'].str.contains(cuisine.lower())) &
        (df['approx_cost(for two people)'] <= max_cost)
    ]

    # Sort by rating and votes (important!)
    filtered = filtered.sort_values(by=['rate', 'votes'], ascending=False)

    return filtered[['name', 'cuisines', 'rate', 'votes', 'approx_cost(for two people)']].head(10)

Testing the function

In [ ]:
recommend_restaurants('Banashankari', 'north indian', 1000)

,name,cuisines,rate,votes,approx_cost(for two people)
480,Ayodhya Upachar,"south indian, north indian, chinese, street food",4.3,734,200.0
634,Ayodhya Upachar,"south indian, north indian, chinese, street food",4.3,734,200.0
34,Faasos,"north indian, biryani, fast food",4.2,415,500.0
694,Faasos,"north indian, biryani, fast food",4.2,415,500.0
2607,Faasos,"north indian, biryani, fast food",4.2,415,500.0
3457,Mini Punjabi Dhaba,north indian,4.2,325,350.0
20086,Mini Punjabi Dhaba,north indian,4.2,296,350.0
191,Mini Punjabi Dhaba,north indian,4.2,287,350.0
556,Mini Punjabi Dhaba,north indian,4.2,287,350.0
2624,Mini Punjabi Dhaba,north indian,4.2,287,350.0


Improving Output Display for Better Visualization

In [ ]:
pd.set_option('display.max_colwidth', None)

Testing the Recommendation Function with Sample Inputs

In [ ]:
recommend_restaurants('Banashankari', 'chinese', 800)
recommend_restaurants('Basavanagudi', 'south indian', 500)

,name,cuisines,rate,votes,approx_cost(for two people)
3335,Brahmin's Coffee Bar,south indian,4.8,2679,100.0
3333,Mavalli Tiffin Room (MTR),south indian,4.5,2896,250.0
769,Vidyarthi Bhavan,south indian,4.4,4460,150.0
3334,Vidyarthi Bhavan,south indian,4.4,4460,150.0
20066,By 2 Coffee,south indian,4.3,319,100.0
2866,By 2 Coffee,south indian,4.3,316,100.0
20496,South Kitchen,south indian,4.3,276,100.0
284,South Kitchen,south indian,4.3,275,100.0
800,South Kitchen,south indian,4.3,275,100.0
3014,South Kitchen,south indian,4.3,275,100.0


Feature Extraction using TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['cuisines'])

Dataset reloading due to session disconnection

In [ ]:
import pandas as pd

df = pd.read_csv('/content/drive/MyDrive/swiggy.csv')

# Keep only useful columns
df = df[['name', 'cuisines']]

# Remove missing values
df.dropna(inplace=True)

# Remove duplicates
df.drop_duplicates(inplace=True)

df.head()

,name,cuisines
0,Jalsa,"North Indian, Mughlai, Chinese"
1,Spice Elephant,"Chinese, North Indian, Thai"
2,San Churro Cafe,"Cafe, Mexican, Italian"
3,Addhuri Udupi Bhojana,"South Indian, North Indian"
4,Grand Village,"North Indian, Rajasthani"


Feature Extraction using TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf = TfidfVectorizer(stop_words='english')

tfidf_matrix = tfidf.fit_transform(df['cuisines'])

Applying cosine similarity

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

Content-Based Restaurant Recommendation using Cosine Similarity

In [ ]:
def recommend(restaurant_name):

    # Get index of restaurant
    idx = df[df['name'] == restaurant_name].index

    if len(idx) == 0:
        print("Restaurant not found")
        return

    idx = idx[0]

    # Get similarity scores
    scores = list(enumerate(cosine_sim[idx]))

    # Sort
    scores = sorted(scores, key=lambda x: x[1], reverse=True)

    # Top 5 recommendations
    scores = scores[1:6]

    print("\nRecommended Restaurants:\n")

    for i in scores:
        print(df.iloc[i[0]]['name'])

Testing the new recommend function

In [ ]:
recommend("Empire Restaurant")


Recommended Restaurants:

Havyaka Mess
Namma Brahmin's Idli
Sri Guru Kottureshwara Davangere Benne Dosa
Sunsadm
Annapooraneshwari Mess


Resetting Dataset Index for Consistency

In [ ]:
df = df.reset_index(drop=True)

Improved Content-Based Recommendation Function using Cosine Similarity

In [ ]:
def recommend_similar(name):

    if name not in df['name'].values:
        return "Restaurant not found"

    idx = df[df['name'] == name].index[0]

    similarity_scores = list(enumerate(cosine_sim[idx]))

    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similarity_scores = similarity_scores[1:11]  # top 10 similar

    restaurant_indices = [i[0] for i in similarity_scores]

    return df.iloc[restaurant_indices][['name', 'cuisines', 'rate']]

Testing the recommendation function

In [ ]:
recommend("Vidyarthi Bhavan")


Recommended Restaurants:

Havyaka Mess
Namma Brahmin's Idli
Sri Guru Kottureshwara Davangere Benne Dosa
Sunsadm
Annapooraneshwari Mess


In [ ]:
recommend("Brahmin's Coffee Bar")


Recommended Restaurants:

Havyaka Mess
Namma Brahmin's Idli
Sri Guru Kottureshwara Davangere Benne Dosa
Sunsadm
Annapooraneshwari Mess


Dropping duplicate columns and checking the updated dataset since dataset had repeatation of hotels

In [ ]:
df = df.drop_duplicates(subset='name')
df = df.reset_index(drop=True)

df.shape

(8785, 2)

Applying TF-IDF and cosine similarity on updated dataset

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(df['cuisines'])

cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

Hybrid Restaurant Recommendation System (Similarity + Ranking)

In [ ]:
def recommend_smart(name, min_rating=3.5):

    if name not in df['name'].values:
        return "Restaurant not found"

    idx = df[df['name'] == name].index[0]

    similarity_scores = list(enumerate(cosine_sim[idx]))
    similarity_scores = sorted(similarity_scores, key=lambda x: x[1], reverse=True)

    similarity_scores = similarity_scores[1:20]  # take more candidates

    restaurant_indices = [i[0] for i in similarity_scores]

    result = df.iloc[restaurant_indices]

    # Apply rating filter
    result = result[result['rate'] >= min_rating]

    # Sort by rating + votes (real-world logic)
    result = result.sort_values(by=['rate', 'votes'], ascending=False)

    return result[['name', 'cuisines', 'rate', 'votes']].head(10)

Testing Hybrid Recommendation

In [ ]:
recommend("Vidyarthi Bhavan")


Recommended Restaurants:

Havyaka Mess
Namma Brahmin's Idli
Sri Guru Kottureshwara Davangere Benne Dosa
Sunsadm
Annapooraneshwari Mess


Model Saving and Serialization using Pickle

In [ ]:
# Save your model + data
import pickle

pickle.dump(df, open('restaurant_data.pkl', 'wb'))
pickle.dump(cosine_sim, open('similarity.pkl', 'wb'))

Food-to-Cuisine Mapping for User Intent Understanding

In [ ]:
food_to_cuisine = {
    "pizza": "italian",
    "pasta": "italian",
    "biryani": "north indian",
    "dosa": "south indian",
    "idli": "south indian",
    "noodles": "chinese",
    "fried rice": "chinese",
    "burger": "fast food"
}

Swiggy-Style Food-Based Restaurant Recommendation System

In [ ]:
def recommend_food_swiggy_style(food, location=None, max_cost=None):

    cuisine = food_to_cuisine.get(food.lower(), food.lower())

    result = df[df['cuisines'].str.contains(cuisine)].copy()

    if location:
        result = result[result['location'].str.lower() == location.lower()]

    if max_cost:
        result = result[result['approx_cost(for two people)'] <= max_cost]

    result.loc[:, 'score'] = (result['rate'] * 2) + (result['votes'] / 1000)

    result = result.sort_values(by='score', ascending=False)

    return result[['name', 'location', 'cuisines', 'rate', 'votes', 'approx_cost(for two people)']].head(10)